# A Latent Diffusion model combining ideas from CS336's language model and a couple image generation papers

Based on:
* The language transformer from CS336 (multihead self attention, adamw, swiglu, rope)
* X-prediction and flow matching from [Let Denoising Generative Models Denoise (Tianhong Li, Kaiming He, et al., 2025)](https://arxiv.org/abs/2511.13720)
* A Vector Quantized latent space from ["Generative Refinement Networks for Visual Synthesis", 
(Han et al., 2026)](https://arxiv.org/abs/2604.13030)

* https://claude.ai/share/9c11840b-4ad9-4736-8119-47e2060b7a02 - GRN -> JiT
* https://claude.ai/chat/4b918b44-b26c-445f-ac9d-7899e60f6875 - Rope based autoencoder vs limited receptive field
* https://share.google/aimode/GQu5a6AvrHfCnCG3G - LPIPS, Euler-Maruyama sampling




In [ ]:
import torch
import hierarchical_binary_quantization.example_narrow_reciptive_field_autoencoder as enrfa
import hierarchical_binary_quantization.example_latent_diffusion_model as eldm
import hierarchical_binary_quantization.misc.checkpoint_helpers as ch
import hierarchical_binary_quantization.misc.dataset_helpers as dh
import hierarchical_binary_quantization.misc.flow_matching_diffusion_helpers as fmdh
import hierarchical_binary_quantization.misc.image_helpers as ih
from hierarchical_binary_quantization.misc.flow_matching_diffusion_helpers import generate_using_direct_x_prediction
from hierarchical_binary_quantization.misc.flow_matching_diffusion_helpers import generate_using_stochastic_heun
from hierarchical_binary_quantization.misc.flow_matching_diffusion_helpers import generate_using_sde
from hierarchical_binary_quantization.misc.flow_matching_diffusion_helpers import generate_simple, generate
import IPython.display as ipd

dataset_name,img_height, img_width = 'fantasy',512,384
grid_height, grid_width = img_height//8, img_width//8

batch_size = 2 # 6 worked for 256x256 before lpips :(
good_autoencoder_checkpoint = '../assets/LocalQuantizingAutoencoder_2026-09-28_23-51-01_ps8,qd16,ld32,body:res,attn,res_3dataset.pth'
good_latent_xpred_diffusion_cp = '../assets/best_so_far_2026-10-03_MinimalSpatialViT_384x512_2026-10-03_08-23-41_3-dataset-autoencoder_cs336_latent_jit.pth.ema.pth'
device='cuda' if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device


In [ ]:
#checkpoint_path = '../../hierarchical-binary-quantization/notebooks/checkpoints/ExampleQuantizingAutoencoder_2026-08-15_12-48-36_.pth'
#autoencoder = hbq_misc.checkpoint_helpers.create_from_checkpoint(ExampleQuantizingAutoencoderWithRope, good_autoencoder_checkpoint)
autoencoder = ch.create_from_checkpoint(enrfa.LocalQuantizingAutoencoder,good_autoencoder_checkpoint)
autoencoder = autoencoder.eval().requires_grad_(False).to(device)

ldm = ch.create_from_checkpoint(eldm.MinimalSpatialViT,good_latent_xpred_diffusion_cp)
ldm = ldm.eval().requires_grad_(False).to(device)

pass

In [ ]:
import re
good_seeds = """12007 303 306 307 12100 11966 11970 29, 277, 11919 11859  169, 249, 11867 12426, 4830, 12379, 12307, 12304, 12244, 12245, 2186, 4, 9, 27, 174, 169, 2103, 2104, 2093, 2082, 2074, 2058, 2059, 2046, 543, 666, 623, 624, 626, 629, 642, 635, 287, 7331, 7376, 641, 12305, 640, 12298, 611, 537, 580, 582, 588, 589, 592, 12228, 11776 609, 612, 622, 545, 553, 552, 556, 558, 562,_71, """
good_seeds = [int(s) for s in re.findall(r'\d+', good_seeds)]


In [ ]:
seed = 0

In [ ]:

if regenerate_examples := True:
    ih.generate_gallery(ldm,autoencoder,n=12900,output_dir="outputs/sample_images")

In [ ]:
def compare_samplers(ldm,seed,steps=30,churn=0.05, noise_scale=1.0):
    torch.manual_seed(seed)
    l1 = generate_simple(ldm,1,16,grid_width=grid_width,grid_height=grid_height,steps=steps,t_eps=0.01, noise_scale=noise_scale)
    torch.manual_seed(seed)
    l2 = generate_using_stochastic_heun(ldm,1,16,grid_width=grid_width,grid_height=grid_height,steps=steps,t_eps=0.01,S_churn=churn, noise_scale=noise_scale-churn)
    torch.manual_seed(seed)
    t_schedule = torch.arange(0.0,1.0,0.1)
    l3 = generate_using_direct_x_prediction(ldm,torch.zeros(2,16,grid_height,grid_width),t_schedule=t_schedule, churn_noise_weight=churn, seed_noise_weight=noise_scale)
    i1,i2,i3 = [autoencoder.decode(autoencoder.post_quant(l)) for l in [l1,l2,l3]]
    p1,p2,p3 = [ih.tensor_to_pil(i[0]) for i in [i1,i2,i3]]
    #print(torch.allclose(l1, l2))
    return ih.html_for_images([p1,p2,p3])


In [ ]:
h = compare_samplers(ldm=ldm,seed=12426,noise_scale=1.0, churn=0.05, steps=20)
ipd.display(ipd.HTML(h))

In [ ]:
h = compare_samplers(ldm=ldm,seed=4,steps=30)
ipd.display(ipd.HTML(h))

In [ ]:
h = compare_samplers(ldm=ldm,seed=9,steps=20)
ipd.display(ipd.HTML(h))

In [ ]:
def compare_churn(ldm,seed,steps=30,churns=[0,0.05,0.2], noise_scales=[1.0,0.95,0.8]):
    torch.manual_seed(seed)
    l1 = generate_using_stochastic_heun(ldm,1,16,grid_width=grid_width,grid_height=grid_height,steps=steps,t_eps=0.01,S_churn=churns[0], noise_scale=noise_scales[0])
    torch.manual_seed(seed)
    l2 = generate_using_stochastic_heun(ldm,1,16,grid_width=grid_width,grid_height=grid_height,steps=steps,t_eps=0.01,S_churn=churns[1], noise_scale=noise_scales[1])
    torch.manual_seed(seed)
    l3 = generate_using_stochastic_heun(ldm,1,16,grid_width=grid_width,grid_height=grid_height,steps=steps,t_eps=0.01,S_churn=churns[2], noise_scale=noise_scales[2])
    i1,i2,i3 = [autoencoder.decode(autoencoder.post_quant(l)) for l in [l1,l2,l3]]
    p1,p2,p3 = [ih.tensor_to_pil(i[0]) for i in [i1,i2,i3]]
    print(torch.allclose(l1, l2))
    return ih.html_for_images([p1,p2,p3])


for idx, seed in enumerate([1,2,3,4,5]):
    h = compare_churn(ldm=ldm,seed=seed,churns=[0,0.05,0.15], noise_scales=[1.0,0.95,0.85], steps=30)
    print(seed)
    ipd.display(ipd.HTML(h))
    if idx > 30:break

In [ ]:
0

In [ ]:
!ls data/dbs/*512

In [ ]:
from torch.utils.data import DataLoader
ds2 = dh.augmented_image_dataset('fantasy',"full_body_portrait",width=384,height=512,src_width=416, src_height=544)
ds2 = dh.augmented_image_dataset('pokemon',"full_body_portrait",width=256,height=256,src_width=256, src_height=256)

idl = DataLoader(ds2, batch_size=1, shuffle=True)
lbls,batches,metas = next(iter(idl))
latent = autoencoder.pre_quant(autoencoder.encode(batches.to(device)))
x = latent
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x)
t_val = 0.95
x_pred = x.clone()
for step in range(100):
    e = torch.randn_like(x)
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = ldm(z, t)
    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    if (step%10)==0:
        print(t_val)
        ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
from hierarchical_binary_quantization.misc.flow_matching_diffusion_helpers import flow_matching_img2img
ds2 = dh.augmented_image_dataset('fantasy',"autoencoder_training",width=384,height=512,src_width=416, src_height=544)
ds2 = dh.augmented_image_dataset('cats_and_dogs',"basic",width=384,height=512,src_width=384, src_height=512)
idl = DataLoader(ds2, batch_size=1, shuffle=True)
lbls,batches,metas = next(iter(idl))
latent = autoencoder.pre_quant(autoencoder.encode(batches.to(device)))
ipd.display(ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(latent))[0]))


In [ ]:
batches.shape

In [ ]:
from PIL import Image, ImageDraw
im = Image.new("RGB", (384,512), (0, 0, 0))
d = ImageDraw.Draw(im)
d.circle((128,64), 32, fill=(128,0,64), outline=None, width=1)
d.ellipse([(128-30,128),(128+30,400)], fill=(256,256,0), outline=None, width=1)
im

In [ ]:
import torchvision.transforms.v2 as v2
import einx
xform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True),v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
batches = einx.id("c h w -> 1 c h w",xform(im))
latent = autoencoder.pre_quant(autoencoder.encode(batches.to(device)))

latent.shape
#            ,

In [ ]:
torch.linspace(0.0,1.0,10)

In [ ]:
#ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant())[0])
x = latent
e = torch.randn_like(x)
t_val = 0.95
x_target = x.clone()
x_pred = x.clone()
for t_val in torch.linspace(0.15,1.0,10):
    #e = torch.randn_like(x)
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    x_pred = (x_pred + x_target)/2
    with torch.no_grad():
        x_pred = ldm(z, t)
    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    print(t_val)
    ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))


In [ ]:
x_ref = latent
result = flow_matching_img2img(ldm, x_ref, steps=30, strength=0.9, method="heun", 
                     t_eps=0.05, noise_scale=1, device="cuda", save_steps=False)

ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(result))[0])


In [ ]:
@torch.no_grad()
def guided_generate_using_stochastic_heun(net, batch_size, latent_dim, grid_height, grid_width, steps, t_eps=0.05,
                                          noise_scale=1.0, device="cuda", S_churn=0.0, S_min=0.0, S_max=0.9, 
                                          save_steps=False, x_ref=None, guidance_strength=0.3,
                                         e_base_input=None):
    """
    Generates an image from pure noise while continuously forcing the trajectory
    to align with the structural layout and color boundaries of x_ref.
    
    Parameters:
        x_ref (Tensor): The encoded HBQ latent of your crayon sketch.
        guidance_strength (float): How strongly to pull the trajectory toward the sketch.
                                   0.0 = Pure generation. 0.5+ = Strong layout lock.
    """
    # Initialize with pure noise at t=0
    #z = noise_scale * torch.randn(batch_size, latent_dim, grid_height, grid_width, device=device)


    # 1. Generate standard high-frequency noise
    e_fresh = torch.randn_like(x_ref) * noise_scale
    
    # 2. Extract the broad structural features of the crayon sketch (Low-Frequency)
    # We do this by applying an adaptive average pool or smooth blur to the latent channel map
    x_ref_low = torch.nn.functional.avg_pool2d(x_ref, kernel_size=5, stride=1, padding=2)
    print(f"min_max {x_ref_low.min()}, {x_ref_low.max()}")
    # 3. Force e_base to structurally align with x_ref
    # This replaces the blind randomness of e_base with noise that shares layout centers
    e_base = e_fresh + x_ref_low*guidance_strength
    print(f"new ebase = {e_base.shape}")

    old_e_base = noise_scale * torch.randn(batch_size, latent_dim, grid_height, grid_width, device=device)
    print(f"old ebase = {old_e_base.shape}")
    guidance_strength=0

    if e_base_input is not None:
        print("no more xref")
        e_base = e_base_input
        x_ref = None
    
    z = e_base.clone()

    if x_ref is not None:
        x_ref = x_ref.to(device)

    t_schedule = torch.linspace(0.0, 1.0, steps + 1, device=device)
    saved_steps = []
    
    for i in range(steps):
        t_cur = t_schedule[i]
        t_next = t_schedule[i + 1]
        
        # --- 1. Stochastic Churn ---
        gamma = 0.0
        if S_min <= t_cur.item() <= S_max:
            gamma = min(S_churn / steps, math.sqrt(2) - 1)
        t_hat = t_cur + gamma * t_cur
        if gamma > 0:
            sigma_fresh = math.sqrt(t_hat**2 - t_cur**2) * noise_scale
            z = z + sigma_fresh * torch.randn_like(z)
            t_cur = t_hat
            
        dt = t_next - t_cur
        
        # --- 2. TRAJECTORY INTERPOLATION GUIDANCE ---
        # If a sketch is provided, we forcefully nudge the current state 'z' to be a linear 
        # blend of its current tracking position and the expected linear path of the sketch.
        if x_ref is not None and guidance_strength > 0.0:
            # We scale the guidance impact by (1 - t) so that it fades out at the end, 
            # letting the LPIPS model finish the eyes/face beautifully without crayon pollution.
            current_fade = guidance_strength * (1.0 - t_cur.item())
            
            # Re-project z slightly toward the sketch's expected trajectory path
            #z_ref_path = t_cur * x_ref + (1.0 - t_cur) * torch.randn_like(z) * noise_scale
            z_ref_path = t_cur * x_ref + (1.0 - t_cur) * e_base
            z = z * (1.0 - current_fade) + z_ref_path * current_fade

        # --- 3. Deterministic Heun Step ---
        t_batch_cur = torch.full((batch_size,), t_cur.item(), device=device)
        x_pred_cur = net(z, t_batch_cur)
        v_cur = (x_pred_cur - z) / (1 - t_cur).clamp_min(t_eps)
        
        z_prime = z + dt * v_cur
        
        if t_next >= 1.0:
            z = z_prime
            if save_steps:
                saved_steps.append(x_pred_cur.cpu().detach())
            continue
            
        t_batch_next = torch.full((batch_size,), t_next.item(), device=device)
        x_pred_next = net(z_prime, t_batch_next)
        v_next = (x_pred_next - z_prime) / (1 - t_next).clamp_min(t_eps)
        
        v_avg = 0.5 * (v_cur + v_next)
        z = z + dt * v_avg
        
        if save_steps:
            saved_steps.append(((x_pred_cur + x_pred_next) / 2.0).cpu().detach())
            
    return (z, saved_steps) if save_steps else z


In [ ]:
def invert_flow(net, x_ref, steps=20, t_eps=0.05):
    device = x_ref.device
    z = x_ref.clone()
    
    # Schedule running backward from 1.0 to 0.0
    t_schedule = torch.linspace(1.0, 0.0, steps + 1, device=device)
    
    for i in range(steps):
        t_cur = t_schedule[i]
        t_next = t_schedule[i + 1]
        dt = t_next - t_cur  # This is negative!
        
        t_batch = torch.full((x_ref.size(0),), t_cur.item(), device=device)
        
        with torch.no_grad():
            x_pred = net(z, t_batch)
            
        # Velocity estimation: v = (x - z) / (1 - t)
        v_pred = (x_pred - z) / (1 - t_cur).clamp_min(t_eps)
        
        # Move backward along the trajectory vector field
        z = z + dt * v_pred
        
    return z

e_base = invert_flow(ldm, x_ref=x_ref, steps=20)
e_fresh = torch.randn_like(e_base)
w0,w1 = 1,20
e_mixed = (e_base*w0 + e_fresh*w1)/math.sqrt(w0*w0+w1*w1)

#ipd.display(ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(e_fresh))[0]))

output_latents = guided_generate_using_stochastic_heun(
    ldm, 
    batch_size=1, 
    latent_dim=16, 
    grid_height=64, 
    grid_width=48, 
    steps=30, 
    x_ref=x_ref, 
    e_base_input=e_mixed,       # Pass this in as your base!
    S_churn=0,
    guidance_strength=0.9
)
ipd.display(ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(output_latents))[0]))



In [ ]:
1/0

# TODO - further explore these other loop ideas

In [ ]:
x.shape

In [ ]:
#lds,ldl = get_latent_dataset_and_dataloader(dataset,shuffle=True, batch_size=1)
x = torch.rand(1,16,grid_height,grid_width).to(device)
x_pred = x.clone()
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x)
for t_val in torch.arange(0.0,1.0,0.1):
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)

    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    print(t_val)
    ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
#lds,ldl = get_latent_dataset_and_dataloader(dataset,shuffle=True, batch_size=1)
x = torch.rand(1,16,grid_height,grid_width).to(device)*2-1
x_pred = x.clone()
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
for t_val in torch.arange(0.0,1.0,0.1):
    e = torch.randn_like(x)
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)

    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    print(t_val)
    ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
#lds,ldl = get_latent_dataset_and_dataloader(dataset,shuffle=True, batch_size=1)
x = torch.rand(1,16,grid_height, grid_width).to(device)*2-1
x_pred = x.clone()
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x) * 1.1
for t_val in torch.arange(0.0,1.0,0.1):
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj._orig_mod(z, t)

    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    print(t_val)
    ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
#lds,ldl = get_latent_dataset_and_dataloader(dataset,shuffle=True, batch_size=1)
x = torch.rand(1,16,grid_height, grid_width).to(device)*2-1
x_pred = x.clone()
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e0w,e1w=0,1
e0 = torch.randn_like(x)
for t_val in torch.arange(0.0,1.0,0.1):
    e1 = torch.randn_like(x)
    e = (e0*e0w + e1*e1w) / (e0w+e1w)**0.5
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)
    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    print(t_val)
    ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
#lds,ldl = get_latent_dataset_and_dataloader(dataset,shuffle=True, batch_size=1)
x = torch.rand(1,16,grid_height, grid_width).to(device)*2-1
x_pred = x.clone()
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e0 = torch.randn_like(x) 
for t_val in torch.arange(0.0,1.0,0.1):
    e1 = torch.randn_like(x)
    e = (e0 + e1) / (1+1)**0.5
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)
    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    print(t_val)
    ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
grid_height, grid_width

## Repeatedly noise and denoise at a given T

In [ ]:
# repeatedly noise and denoise at a given T


lds,ldl = get_latent_dataset_and_dataloader("fantasy",width=img_width, height=img_height, shuffle=True, batch_size=1)
x = next(iter(ldl))[1].to(device)
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x)
t_val = 0.975
x_pred = x.clone()
for step in range(100):
    e = torch.randn_like(x)
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)
    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    if (step%10)==0:
        print(t_val)
        ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
# repeatedly noise and denoise at a given T


lds,ldl = get_latent_dataset_and_dataloader("fantasy",width=img_width, height=img_height, shuffle=True, batch_size=1)
x = next(iter(ldl))[1].to(device)
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x)
t_val = 0.75
x_pred = x.clone()
for step in range(100):
    e = torch.randn_like(x)
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)
    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    if (step%10)==0:
        print(t_val)
        ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
# repeatedly noise and denoise at a given T


#lds,ldl = get_latent_dataset_and_dataloader("fantasy",width=img_width, height=img_height, shuffle=True, batch_size=1)
#x = next(iter(ldl))[1].to(device)
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x)
t_val = 0.95
x_pred = x.clone()
for step in range(100):
    e = torch.randn_like(x)
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)
    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    if (step%10)==0:
        print(t_val)
        ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
# repeatedly noise and denoise at a given T


lds,ldl = get_latent_dataset_and_dataloader("fantasy",width=img_width, height=img_height, shuffle=True, batch_size=1)
x = next(iter(ldl))[1].to(device)
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x)
t_val = 0.95
x_pred = x.clone()
for step in range(100):
    e = torch.randn_like(x)
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)
    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    if (step%10)==0:
        print(t_val)
        ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
# repeatedly noise and denoise at a given T


lds,ldl = get_latent_dataset_and_dataloader("fantasy",width=img_width, height=img_height, shuffle=True, batch_size=1)
x = next(iter(ldl))[1].to(device)
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x)
t_val = 0.95
x_pred = x.clone()
for step in range(100):
    e = torch.randn_like(x)
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)
    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    if (step%10)==0:
        print(t_val)
        ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
# repeatedly noise and denoise at a given T


lds,ldl = get_latent_dataset_and_dataloader("fantasy",width=img_width, height=img_height, shuffle=True, batch_size=1)
x = next(iter(ldl))[1].to(device)
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x)
t_val = 0.69
x_pred = x.clone()
for step in range(100):
    e = torch.randn_like(x)
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)
    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    if (step%10)==0:
        print(t_val)
        ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
# repeatedly noise and denoise at a given T


lds,ldl = get_latent_dataset_and_dataloader("fantasy",width=img_width, height=img_height, shuffle=True, batch_size=1)
x = next(iter(ldl))[1].to(device)
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x)
t_val = 0.69
x_pred = x.clone()
for step in range(100):
    e = torch.randn_like(x)
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)
    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred))[0])
    if (step%10)==0:
        print(t_val)
        ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))